# Geison - Assay discovery workbench

A guided, auditable qPCR assay-discovery workflow for a researcher. It separates target conservation, target-vs-non-target contrast, oligo design, target coverage, and final specificity. The notebook only operates the published `qpcr-pipeline` CLI and visualizes its artifacts; scientific calculations remain in Geison.

> The bundled demo is synthetic and educational. Candidate assays require independent review and wet-lab validation.

## 0. Setup
Install Geison from `main`, install its command-line tools, record the commit, and inspect environment readiness.

In [ ]:
%%bash
set -euo pipefail
cd /content
if [ -d Geison/.git ]; then git -C Geison checkout main && git -C Geison pull --ff-only origin main; else git clone --branch main https://github.com/BrunoDCamargo/Geison.git; fi
python -m pip install -q -e /content/Geison pandas matplotlib pyyaml
apt-get update -qq && DEBIAN_FRONTEND=noninteractive apt-get install -y -qq cd-hit mafft primer3
git -C /content/Geison rev-parse HEAD
cd /content/Geison && qpcr-pipeline doctor


## 1. Project and panel
Choose **Demo (synthetic)** for a complete reproducible walkthrough or **Project** for local FASTA inputs. Project mode accepts one target FASTA and up to five challenge datasets. Names must match the panel entries exactly. `CRITICAL` challenges control the first contrast ordering key; `IMPORTANT` comes next; no hidden pass threshold is applied.

In [ ]:
from pathlib import Path
import html, json, shlex, subprocess, yaml
from IPython.display import HTML, Markdown, display
mode = "Demo (synthetic)" #@param ["Demo (synthetic)", "Project"]
workspace = "/content/geison_workbench" #@param {type:"string"}
project_target_name = "My target" #@param {type:"string"}
project_target_fasta = "/content/target.fasta" #@param {type:"string"}
challenge_1_name = "" #@param {type:"string"}
challenge_1_fasta = "" #@param {type:"string"}
challenge_1_criticality = "CRITICAL" #@param ["CRITICAL", "IMPORTANT", "BACKGROUND"]
challenge_2_name = "" #@param {type:"string"}
challenge_2_fasta = "" #@param {type:"string"}
challenge_2_criticality = "IMPORTANT" #@param ["CRITICAL", "IMPORTANT", "BACKGROUND"]
challenge_3_name = "" #@param {type:"string"}
challenge_3_fasta = "" #@param {type:"string"}
challenge_3_criticality = "BACKGROUND" #@param ["CRITICAL", "IMPORTANT", "BACKGROUND"]
challenge_4_name = "" #@param {type:"string"}
challenge_4_fasta = "" #@param {type:"string"}
challenge_4_criticality = "BACKGROUND" #@param ["CRITICAL", "IMPORTANT", "BACKGROUND"]
challenge_5_name = "" #@param {type:"string"}
challenge_5_fasta = "" #@param {type:"string"}
challenge_5_criticality = "BACKGROUND" #@param ["CRITICAL", "IMPORTANT", "BACKGROUND"]


In [ ]:
project_dir = Path(workspace); project_dir.mkdir(parents=True, exist_ok=True)
output_dir = project_dir / "output"
if mode == "Demo (synthetic)":
    subprocess.run(["python", "/content/Geison/examples/guided_demo/generate_demo_data.py", str(project_dir)], check=True)
else:
    slots = [(challenge_1_name, challenge_1_fasta, challenge_1_criticality), (challenge_2_name, challenge_2_fasta, challenge_2_criticality), (challenge_3_name, challenge_3_fasta, challenge_3_criticality), (challenge_4_name, challenge_4_fasta, challenge_4_criticality), (challenge_5_name, challenge_5_fasta, challenge_5_criticality)]
    challenges = [(n.strip(), p.strip(), c) for n, p, c in slots if n.strip() and p.strip()]
    if not challenges: raise ValueError("Project mode requires at least one challenge name/path pair.")
    definition = {"target": {"name": project_target_name, "taxid": None, "mode": "broad_detection", "subtype": None, "groups": [{"name": "project target set", "required": True, "dataset_roles": ["DESIGN"], "reasons": ["researcher supplied"], "proposed_by": ["guided workbench"], "sequence_selection": []}]}, "non_targets": [{"name": n, "taxid": None, "criticality": c, "dataset_roles": ["CHALLENGE"], "reasons": ["researcher supplied differential"], "proposed_by": ["guided workbench"], "sequence_selection": []} for n,p,c in challenges], "diagnostic_context": {"syndrome": None, "geography": None, "sample_type": None, "vector": None}}
    base = {"target": {"name": project_target_name}, "input": {"fasta": project_target_fasta}, "alignment": {"enabled": True, "threads": 2}, "conservation": {"enabled": True, "window_size": 100, "step_size": 20}, "primer_design": {"enabled": True, "max_candidate_regions": 6, "assays_per_region": 3}, "inclusivity": {"enabled": True}, "off_targets": [{"name": n, "fasta": p} for n,p,c in challenges], "specificity": {"enabled": True}, "ranking": {"enabled": True}}
    proposal = dict(base); proposal["panel"] = {"proposal": definition}; proposal["contrastive_conservation"] = {"enabled": False}
    approved = dict(base); approved["panel"] = {"frozen_manifest": "__APPROVED_PANEL_PATH__"}; approved["contrastive_conservation"] = {"enabled": True}
    (project_dir / "config-proposal.yaml").write_text(yaml.safe_dump(proposal, sort_keys=False), encoding="utf-8")
    (project_dir / "config-approved-template.yaml").write_text(yaml.safe_dump(approved, sort_keys=False), encoding="utf-8")
display(Markdown("### Generated proposal configuration")); print((project_dir / "config-proposal.yaml").read_text())


## 2. Data readiness
The first run intentionally stops at `ACTION_REQUIRED / PANEL_APPROVAL_REQUIRED`. Review the exact frozen scientific panel before any sequence analysis begins.

In [ ]:
proposal_config = project_dir / "config-proposal.yaml"
first = subprocess.run(["qpcr-pipeline", "run", str(proposal_config), "--outdir", str(output_dir)], cwd=project_dir, capture_output=True, text=True)
print(first.stdout); print(first.stderr if first.stderr else "")
if first.returncode != 3 or "PANEL_APPROVAL_REQUIRED" not in first.stdout: raise RuntimeError("Expected ACTION_REQUIRED / PANEL_APPROVAL_REQUIRED before scientific execution.")
proposal_path = output_dir / "panel_proposal.yaml"
proposal_text = proposal_path.read_text(encoding="utf-8")
display(HTML("<div style='border:2px solid #167d6b;border-radius:14px;padding:18px;background:#f5fbf8'><h3>Panel review card</h3><p>Check target groups, every challenge organism, criticality, and rationale.</p><pre style='white-space:pre-wrap'>" + html.escape(proposal_text) + "</pre></div>"))


## 3. Explicit panel approval
Enter `APROVAR` only after scientific review. Approval creates an immutable manifest and the approved configuration remains visible for audit.

In [ ]:
approval = "" #@param {type:"string"}
if approval != "APROVAR": raise RuntimeError("Approval paused. Review the panel and enter APROVAR.")
approved_panel_path = project_dir / "approved_panel.json"
approved_run = subprocess.run(["qpcr-pipeline", "panel", "approve", str(proposal_path), "--output", str(approved_panel_path)], capture_output=True, text=True)
print(approved_run.stdout); print(approved_run.stderr if approved_run.stderr else "")
if approved_run.returncode: raise RuntimeError("qpcr-pipeline panel approve failed")
template = yaml.safe_load((project_dir / "config-approved-template.yaml").read_text())
template["panel"]["frozen_manifest"] = str(approved_panel_path)
approved_config = project_dir / "config-approved.yaml"
approved_config.write_text(yaml.safe_dump(template, sort_keys=False), encoding="utf-8")
display(Markdown("### Generated approved configuration")); print(approved_config.read_text())
resumed = subprocess.run(["qpcr-pipeline", "run", str(approved_config), "--outdir", str(output_dir), "--resume"], cwd=project_dir, capture_output=True, text=True)
print(resumed.stdout); print(resumed.stderr if resumed.stderr else "")
if resumed.returncode: raise RuntimeError("Approved qpcr-pipeline run --resume failed")


## 4. Run state
Status is read from `run_manifest.json`, never inferred from the presence of a chart. `ACTION_REQUIRED`, `PARTIAL`, and `FAILED` are not completion.

In [ ]:
manifest_path = output_dir / "run_manifest.json"
run_manifest = json.loads(manifest_path.read_text())
status = run_manifest.get("status")
if status == "ACTION_REQUIRED": message = "Review and approve the panel before scientific execution."
elif status == "PARTIAL": message = "PARTIAL: missing_evidence = " + json.dumps(run_manifest.get("completeness", {}).get("missing_evidence", []))
elif status == "FAILED": message = "FAILED: stage = " + str(run_manifest.get("failure", {}).get("stage")) + "; diagnostic = " + str(run_manifest.get("failure", {}).get("message", "see manifest"))
elif status == "COMPLETED": message = "COMPLETED: all required evidence recorded in run_manifest.json."
else: message = "Run state: " + str(status)
display(HTML("<div style='padding:16px;border-radius:12px;background:#edf7f2;font-weight:700'>" + html.escape(message) + "</div>"))


## 5. Target conservation
These are target-only metrics from the published conservation artifacts. High conservation says the target population agrees; it does not yet say that nearby non-targets differ.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
conservation_report = json.loads((output_dir / "conservation/conservation_report.json").read_text())
conservation_windows = pd.read_csv(output_dir / "conservation/window_metrics.tsv", sep="\t")
display(conservation_windows.sort_values(["mean_conservation", "mean_entropy_bits"], ascending=[False, True]).head(12))
plt.figure(figsize=(12,3)); plt.plot((conservation_windows.reference_start + conservation_windows.reference_end)/2, conservation_windows.mean_conservation, color="#167d6b"); plt.ylim(0,1.02); plt.xlabel("Reference position"); plt.ylabel("Target conservation"); plt.grid(alpha=.2); plt.show()


## 6. Target vs non-target contrast
The X axis is the strongest local similarity observed in an approved challenge dataset; lower is more discriminant. The Y axis is target conservation; higher is more stable. This view reads `contrastive_conservation/window_metrics.tsv` and does not recompute similarity or ranking.

In [ ]:
contrast_windows = pd.read_csv(output_dir / "contrastive_conservation/window_metrics.tsv", sep="\t")
contrast_candidates = pd.read_csv(output_dir / "contrastive_conservation/candidate_regions.tsv", sep="\t")
dataset_metrics = pd.read_csv(output_dir / "contrastive_conservation/dataset_metrics.tsv", sep="\t")
fig, axes = plt.subplots(1,2,figsize=(14,4)); axes[0].scatter(contrast_windows.worst_similarity, contrast_windows.target_mean_conservation, c=contrast_windows.target_eligible.map({True:"#167d6b",False:"#9aa7a3"})); axes[0].set(xlabel="Worst challenge similarity (lower is better)", ylabel="Target conservation (higher is better)", title="Contrast quadrant"); axes[0].grid(alpha=.2)
mid=(contrast_windows.reference_start+contrast_windows.reference_end)/2; axes[1].plot(mid, contrast_windows.target_mean_conservation,label="target conservation",color="#167d6b"); axes[1].plot(mid, contrast_windows.worst_similarity,label="worst challenge",color="#d6624b"); axes[1].set(xlabel="Reference position",ylabel="Score",title="Reference track"); axes[1].legend(); axes[1].grid(alpha=.2); plt.show()
display(Markdown("### Selected contrast regions")); display(contrast_candidates)
display(Markdown("### Per-dataset evidence")); display(dataset_metrics)
display(HTML((output_dir / "contrastive_conservation/report.html").read_text(encoding="utf-8")))


## 7. Assay design
Primer3 receives exactly the contrast-selected regions. Inspect design counts and oligo geometry; these are candidates, not validated assays.

In [ ]:
primer_report = json.loads((output_dir / "primer_design/primer_design_report.json").read_text())
print("Candidate source:", primer_report.get("candidate_source")); print("Counts:", primer_report.get("counts"))
assays_path = output_dir / "primer_design/assays.tsv"
display(pd.read_csv(assays_path, sep="\t") if assays_path.exists() else Markdown("No assay table was published."))


## 8. Target coverage
Inclusivity asks whether each candidate covers the evaluation target set, including mismatch and degeneracy evidence.

In [ ]:
inclusivity_report = json.loads((output_dir / "inclusivity/inclusivity_report.json").read_text())
print(json.dumps({"status": inclusivity_report.get("status"), "counts": inclusivity_report.get("counts")}, indent=2))
coverage_path = output_dir / "inclusivity/assay_inclusivity.tsv"
if coverage_path.exists(): display(pd.read_csv(coverage_path, sep="\t"))


## 9. Specificity
Final specificity is assay-level evidence based on primer/probe hits and plausible amplicon geometry. It is distinct from region-level contrast.

In [ ]:
specificity_report = json.loads((output_dir / "specificity/specificity_report.json").read_text())
print(json.dumps({"status": specificity_report.get("status"), "counts": specificity_report.get("counts")}, indent=2))
amplicons_path = output_dir / "specificity/plausible_amplicons.tsv"
if amplicons_path.exists(): display(pd.read_csv(amplicons_path, sep="\t"))


## 10. Final candidates
The final table combines conservation, inclusivity, Primer3 quality, robustness, and specificity under explicit reason codes. Only `run_manifest.json: COMPLETED` permits presenting the run as complete.

In [ ]:
ranking_report = json.loads((output_dir / "ranking/ranking_report.json").read_text())
ranking_path = output_dir / "ranking/assay_ranking.tsv"
display(pd.read_csv(ranking_path, sep="\t") if ranking_path.exists() else Markdown("No final candidates were published. Inspect missing_evidence above."))


## 11. Reproducibility and Advanced evidence
Preserve both generated YAML files, `approved_panel.json`, `run_manifest.json`, and the complete output directory. Advanced inspection below exposes tool versions, commit, checkpoints, and raw artifact paths.

In [ ]:
commit = subprocess.run(["git", "-C", "/content/Geison", "rev-parse", "HEAD"], capture_output=True, text=True, check=True).stdout.strip()
checkpoint_files = sorted(str(path.relative_to(output_dir)) for path in (output_dir / ".checkpoints").glob("*/manifest.json"))
advanced = {"commit": commit, "status": run_manifest.get("status"), "environment": run_manifest.get("environment"), "checkpoints": checkpoint_files, "raw_artifacts": ["conservation/window_metrics.tsv", "contrastive_conservation/window_metrics.tsv", "contrastive_conservation/candidate_regions.tsv", "contrastive_conservation/dataset_metrics.tsv", "primer_design/primer_design_report.json", "specificity/specificity_report.json", "ranking/ranking_report.json"]}
display(HTML("<details><summary><b>Advanced: generated configs and evidence</b></summary><h4>config-proposal.yaml</h4><pre>" + html.escape(proposal_config.read_text()) + "</pre><h4>config-approved.yaml</h4><pre>" + html.escape(approved_config.read_text()) + "</pre><h4>run_manifest.json</h4><pre>" + html.escape(json.dumps(run_manifest, indent=2)) + "</pre><h4>Evidence index</h4><pre>" + html.escape(json.dumps(advanced, indent=2)) + "</pre></details>"))
